In [ ]:
# What, substantively, are the differences between old and new frames? 
# Is there an actual differencen in values? Can I reconstruct the neighbor graph that the old one seemed to produce



In [ ]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd 
import scipy


# from pydeseq2.dds import DeseqDataSet
# from pydeseq2.default_inference import DefaultInference
# from pydeseq2.ds import DeseqStats

from matplotlib.colors import LinearSegmentedColormap
import matplotlib.cm as cm

from misc_utils import score_colors

In [ ]:
data_path = "/Users/bbrener1/haxx/ceisel_mumm/data/"
# new = sc.read_h5ad(data_path + "full_annotations_leiden_new.h5ad")
new = sc.read_h5ad(data_path + "joint_annotated_new_neighbors.h5ad")
old = sc.read_h5ad(data_path + "full_annotations_leiden_old.h5ad")



# Mismatch Verification

In [ ]:
print(new.shape)
print(old.shape)

In [ ]:
np.sum(new.obs_names == old.obs_names)
barcode_mismatch_mask = new.obs_names != old.obs_names
print(new[barcode_mismatch_mask].obs)

In [ ]:
old_rgcs = old.obs['cell_type'] == 'RGCs'
in_new = new[old_rgcs]
new_rgcs = new.obs['cell_type'] == 'RGCs'
in_old = old[new_rgcs]

print(in_new.obs['cell_type'])
print(in_old.obs['cell_type'])

In [ ]:
print(np.sum(old.obs['cell_type']=="RGCs"))
print(np.sum(new.obs['cell_type']=="RGCs"))

# Value comparison

In [ ]:
flat_random_mask = np.random.rand(new.shape[0]*new.shape[1]) < 0.01
plt.figure()
plt.scatter(new.X.toarray().flatten()[flat_random_mask],old.X.toarray().flatten()[flat_random_mask],s=1)
plt.show()

In [ ]:
flat_random_mask = np.random.rand(new.shape[0]*new.obsm['X_pca'].shape[1]) < 0.01
plt.figure()
plt.scatter(
    new.obsm['X_pca'].flatten(),
    old.obsm['X_pca'].flatten(),
    s=1
)
plt.show()


In [ ]:
# Psych, reference diagonals aren't good enough lol
print(np.allclose(np.abs(old.obsm['X_pca']),np.abs(new.obsm['X_pca']),atol=1e-4))
print(np.allclose(np.abs(old.varm['PCs']),np.abs(new.varm['PCs'])))


In [ ]:
delta = np.abs(old.obsm['X_pca']) - np.abs(new.obsm['X_pca'])

plt.figure()
plt.hist(delta.flatten(),bins=50,log=True)
plt.show()
# np.where(np.abs(old.obsm['X_pca'] - new.obsm['X_pca']) > 1e-6)

In [ ]:
# Let's try a couple of the explicit solvers to see what happens
# not arpack 
old.X = old.X.toarray()

old_old_pca = old.obsm['X_pca'].copy()

del(old.obsm)
sc.pp.pca(old,svd_solver='arpack')
old_new_pca = old.obsm['X_pca'].copy()

# plt.figure()
# plt.scatter(old_old_pca.flatten(),old_new_pca.flatten(),s=1)
# plt.show()

delta = np.abs(old_old_pca) - np.abs(old_new_pca)

plt.figure()
plt.hist(delta.flatten(),bins=50,log=True)
plt.show()


In [ ]:
# x_x ok yes this is run to run variance 
# But what if that's enough? Let's re-compute up to PCA
# It appears that re-computing the PCA is sufficient to alter the distances a similar amount
# Hm
# But there is a big change happening here. 

# Neighbors comparison 

In [ ]:
old_old_distances = old.obsp['distances'].copy()

In [ ]:
del(old.obsp)
# del(old.obsm)
# sc.pp.pca(old)
sc.pp.neighbors(old)

In [ ]:
new_old_distances = old.obsp['distances'].copy()

In [ ]:
random_mask = np.random.rand(old.shape[0]) < 0.001
plt.figure()
plt.scatter(
    old_old_distances[random_mask].toarray().flatten(),
    new_old_distances[random_mask].toarray().flatten(),
    # old.obsp['distances'][random_mask].toarray().flatten(),
    # new.obsp['distances'][random_mask].toarray().flatten(),
    s=1
)
plt.show()

# plt.figure()
# plt.scatter(
#     old.obsp['connectivities'][random_mask].toarray().flatten(),
#     new.obsp['connectivities'][random_mask].toarray().flatten(),
#     s=1
# )
# plt.show()


In [ ]:
old.obsp['connectivities']

# Leiden comparison

In [ ]:
old.obsm['X_pca']

In [ ]:
# We end up with a distance mismatch even if I recompute the old frame with the current env. But I KNOW the current env produces identical values + pcas? ???
# What else is even an input in there? Neighbors object not deleted and it has params?

# General Delta Notes

In [ ]:
old

In [ ]:
new

In [ ]:
# Diff notes:
# connectivities and distances are in opposite order in obsp, I guess.
# both have a raw layer
# both have varm PCs (identical?)

In [ ]:
old.uns['pca']

In [ ]:
new.uns['pca']

In [ ]:
# CORE FINDINGS:
# There is a small but non-negligible difference in the PCA solution, seems to be driving the observed differences, not actually the UMAP (although that may be invovled too)

# Plotting the diagonal is not sufficient, plot the delta hist between versions 
# Not run-to-run variance in PCA solution, seems to be version diff


# Densifying and solving PCA again doesn't seem to work 
# 